In [5]:
import os
import chromadb

from groq import Groq

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from dotenv import load_dotenv

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv('GROQ_API_KEY')

# Define the client for groq
client = Groq(api_key=os.environ["GROQ_API_KEY"])

# Define model name
model_name = 'openai/gpt-oss-120b'
tesla_10k_collection = 'tesla-10k-2019-to-2023'

chromadb_client = chromadb.PersistentClient(
    path="./tesla_db"
)

hyde_collection = chromadb_client.get_collection(
    name="tesla-10k-2019-to-2023-hyde"
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


In [7]:
# Making the embedding model
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

In [8]:
vectorstore_persisted = Chroma(
    collection_name=tesla_10k_collection,
    collection_metadata={"hsnw:space": "cosine"},
    embedding_function=embedding_model,
    client=chromadb_client,
    persist_directory="./tesla_db"
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [9]:
retriever = vectorstore_persisted.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 5}
)

In [10]:
collection = vectorstore_persisted._collection

total_docs = collection.count()

print(total_docs)

3336


### **Retrieval**

In [11]:
def baseline_retrieve(
    query
):
    return "\n---\n".join([retriever.invoke(
        query
    )[i].page_content for i in range(5)])

In [12]:
baseline_docs = baseline_retrieve(
    "What risks could affect Tesla's ability to scale production?"
)

baseline_docs[:1000]

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


'inventory.\tSimilarly,\tthe\tincreasing\tnumber\tof\tTesla\tvehicles\talso\trequires\tus\tto\tcontinue\tto\trapidly\tincrease\tthe\tnumber\tof\tour\tSupercharger\tstations\tand\nconnectors\tthroughout\tthe\tworld.\nThere\tis\tno\tassurance\tthat\twe\twill\tbe\table\tto\tramp\tour\tbusiness\tto\tmeet\tour\tsales,\tdelivery,\tinstallation,\tservicing\tand\tvehicle\tcharging\ttargets\tglobally,\nthat\tour\tprojections\ton\twhich\tsuch\ttargets\tare\tbased\twill\tprove\taccurate\tor\tthat\tthe\tpace\tof\tgrowth\tor\tcoverage\tof\tour\tcustomer\tinfrastructure\tnetwork\twill\nmeet\tcustomer\texpectations.\t\nThese\tplans\trequire\tsignificant\tcash\tinvestments\tand\tmanagement\tresources\tand\tthere\tis\tno\tguarantee\tthat\tthey\twill\tgenerate\nadditional\tsales\tor\tinstallations\tof\tour\tproducts,\tor\tthat\twe\twill\tbe\table\tto\tavoid\tcost\toverruns\tor\tbe\table\tto\thire\tadditional\tpersonnel\tto\tsupport\tthem.\t\nAs\twe\nexpand,\tw\ne\twill\talso\tneed\tto\tensure\tour\tcomp

In [57]:
def get_hyde_questions_chunks(
    query,
    k=5
):
    hyde_results = hyde_collection.query(
        query_texts=[query],
        n_results=k
    )

    return [hyde_results["ids"][0], hyde_results["documents"][0]]

In [58]:
get_hyde_questions_chunks("What risks could affect Tesla's ability to scale production?")

[['text_214', 'text_204', 'text_912', 'text_2267', 'text_916'],
 ["What are the potential risks and challenges associated with Tesla's growth and expansion plans? How might Tesla's ability to manage its growth effectively impact its business and financial condition? What are the potential consequences of Tesla's failure to meet its sales, delivery, and servicing targets?",
  "What are the risks associated with Tesla's business and industry? How may delays or complications in production impact Tesla's business? What are the potential consequences of Tesla not meeting its manufacturing cost targets?",
  "What are the potential risks to Tesla's business operations mentioned in the text? How may global trade conditions and consumer trends impact Tesla's business? What are some of the challenges that Tesla may face in sustaining its production trajectory?",
  "What are some challenges that Tesla may face in launching and ramping up production of its products? How does Tesla plan to achieve 

In [61]:
def hyde_retrieve(
    query,
    k=5
):
    parent_ids = get_hyde_questions_chunks(query, k=k)[0]
    questions = get_hyde_questions_chunks(query, k=k)[1]
    original_docs = vectorstore_persisted.get(
        ids=parent_ids
    )

    return [parent_ids, "\n---\n".join(original_docs['documents']), questions]

In [60]:
hyde_docs = hyde_retrieve("What risks could affect Tesla's ability to scale production?")[2]
hyde_docs

["What are the potential risks and challenges associated with Tesla's growth and expansion plans? How might Tesla's ability to manage its growth effectively impact its business and financial condition? What are the potential consequences of Tesla's failure to meet its sales, delivery, and servicing targets?",
 "What are the risks associated with Tesla's business and industry? How may delays or complications in production impact Tesla's business? What are the potential consequences of Tesla not meeting its manufacturing cost targets?",
 "What are the potential risks to Tesla's business operations mentioned in the text? How may global trade conditions and consumer trends impact Tesla's business? What are some of the challenges that Tesla may face in sustaining its production trajectory?",
 "What are some challenges that Tesla may face in launching and ramping up production of its products? How does Tesla plan to achieve efficient and cost-effective manufacturing capabilities? What are so

### **Response Generation and retrieval**

In [29]:
query_system_message = """
You are an assistant to a financial services firm who answers user queries on annual reports.
User input will have the context required by you to answer user queries.
This context will be delimited by: <Context> and </Context>.
The context contains references to specific portions of a document relevant to the user query.

User queries will be delimited by: <Question> and </Question>.

Please answer user queries only using the context provided in the input.
Do not mention anything about the context in your final answer. Your response should only contain the answer to the question.

If the answer is not found in the context, respond "I don't know".
"""

In [30]:
query_user_message_template = """
<Context>
Here are some documents that are relevant to the question mentioned below.
{context}
</Context>

<Question>
{question}
</Question>
"""

In [31]:
def answer_question(
    query,
    docs
):

    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {
                "role": "system",
                "content": query_system_message
            },
            {
                "role": "user",
                "content": query_user_message_template.format(
                    context=docs,
                    question=query
                )
            }
        ]
    )

    return response.choices[0].message.content

### **Benchmark Questions**

In [35]:
queries = {
    "HQ1": "What should a board member ask about risks that could prevent Tesla from meeting production, delivery, or scaling expectations?",
    "HQ2": "How should an analyst investigate the relationship between product defects, warranty or service obligations, customer trust, and brand risk?",
    "HQ3": "What evidence helps determine whether future cash flow depends more on capital expenditure discipline, working capital, or operating income?",
    "HQ4": "Which disclosures help evaluate technology, cybersecurity, data, or AI operational risk even if the user does not explicitly say “cybersecurity”?"
}

In [64]:
model_name = 'openai/gpt-oss-20b'

In [65]:
import json
import traceback
from pathlib import Path

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

results = {}

for qid, query in queries.items():

    print("\n" + "=" * 80)
    print(f"Processing {qid}")
    print("=" * 80)

    try:

        # --------------------------
        # Baseline Retrieval
        # --------------------------

        print("Running baseline retrieval...")

        baseline_docs = baseline_retrieve(query)

        print(f"Retrieved {len(baseline_docs)} baseline docs")

        # --------------------------
        # HyDE Retrieval
        # --------------------------

        print("Running HyDE retrieval...")

        hyde_docs = hyde_retrieve(query)[1]

        print(f"Retrieved {len(hyde_docs)} HyDE docs")
        
        # --------------------------
        # Answer Generation
        # --------------------------

        print("Generating baseline answer...")

        baseline_answer = answer_question(
            query,
            baseline_docs
        )

        print("Generating HyDE answer...")

        hyde_answer = answer_question(
            query,
            hyde_docs
        )
        comparison_with_baseline = f"""
        Baseline retrieved {len(baseline_docs)} chunks.

        HyDE retrieved {len(hyde_docs)} chunks.

        HyDE retrieval was used for the final answer.
        """
        # --------------------------
        # Store Results
        # --------------------------

        result = {
            "question_id": qid,
            "query": query,
            "retrieved_hypothetical_questions": [
                {
                    "parent_chunk_ids": hyde_retrieve(query)[0],
                    "hypothetical_questions": hyde_retrieve(query)[2]
                }
            ],
            "baseline_answer": baseline_answer,
            "hyde_answer": hyde_answer,
            "baseline_chunks": baseline_docs,
            "hyde_chunks": hyde_docs,
            "final_answer": hyde_answer,
            "comparison_with_baseline": comparison_with_baseline
        }

        results[qid] = result

        # save immediately
        output_file = OUTPUT_DIR / f"{qid}.json"

        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(result, f, indent=2, ensure_ascii=False)

        print(f"{qid} ✓ Saved")

    except Exception as e:

        print(f"{qid} ✗ Failed")
        print(f"Error: {e}")

        traceback.print_exc()

        # save failure log
        with open(
            OUTPUT_DIR / "failures.log",
            "a",
            encoding="utf-8"
        ) as f:

            f.write("\n" + "=" * 80 + "\n")
            f.write(f"{qid}\n")
            f.write(f"Query:\n{query}\n\n")
            f.write(f"Error:\n{str(e)}\n\n")
            f.write(traceback.format_exc())
            f.write("\n")

        continue

print("\nFinished evaluation.")


Processing HQ1
Running baseline retrieval...
Retrieved 7004 baseline docs
Running HyDE retrieval...
Retrieved 8048 HyDE docs
Generating baseline answer...
Generating HyDE answer...
HQ1 ✓ Saved

Processing HQ2
Running baseline retrieval...
Retrieved 6843 baseline docs
Running HyDE retrieval...
Retrieved 8422 HyDE docs
Generating baseline answer...
Generating HyDE answer...
HQ2 ✓ Saved

Processing HQ3
Running baseline retrieval...
Retrieved 4645 baseline docs
Running HyDE retrieval...
Retrieved 6303 HyDE docs
Generating baseline answer...
Generating HyDE answer...
HQ3 ✓ Saved

Processing HQ4
Running baseline retrieval...
Retrieved 8276 baseline docs
Running HyDE retrieval...
Retrieved 7703 HyDE docs
Generating baseline answer...
Generating HyDE answer...
HQ4 ✓ Saved

Finished evaluation.
